# W4111 API smoke tests (`app/main.py`)

This notebook calls **`classicmodels`** routes (**`/customers`**, **`/orders`**, **`/orderdetails`**) over HTTP with **`requests`**. Optional cells at the end exercise the starter **`/harry-potter`** sample (JSON file).

**Prerequisite:** Load MySQL **`classicmodels`** and create **`.env`** (see **`.env.example`**). Run the API from the repo root, e.g. **`python -m app.main`** or **`uvicorn app.main:app --reload --port 8000`**. Default base URL **`http://127.0.0.1:8000`** — override with **`API_BASE_URL`** if needed.

**Writes:** **`POST`**, **`PUT`**, **`DELETE`** cells change MySQL data. Run mutating cells cautiously on a shared database.

In [ ]:
import os

import requests

BASE_URL = os.environ.get("API_BASE_URL", "http://127.0.0.1:8000").rstrip("/")

try:
    _health = requests.get(f"{BASE_URL}/health", timeout=5)
    _health.raise_for_status()
except requests.RequestException as exc:
    raise RuntimeError(
        f"Cannot reach API at {BASE_URL!r}. Start the server from the repo root, then rerun this cell."
    ) from exc

print("OK:", BASE_URL)

## `GET /customers` — list / filter (equality query parameters)

In [ ]:
resp = requests.get(f"{BASE_URL}/customers", timeout=30)
assert resp.status_code == 200, resp.text
items = resp.json()["items"]
assert len(items) > 0
SAMPLE_CUSTOMER_NUMBER = items[0]["customerNumber"]
items[:2]

## `GET /customers/{customerNumber}` — `404` when missing

In [ ]:
r = requests.get(f"{BASE_URL}/customers/{SAMPLE_CUSTOMER_NUMBER}", timeout=30)
assert r.status_code == 200
r.json()

In [ ]:
missing = requests.get(f"{BASE_URL}/customers/99999999", timeout=30)
assert missing.status_code == 404
missing.json()

## `GET /orders` and `GET /orders/{orderNumber}`

In [ ]:
resp = requests.get(f"{BASE_URL}/orders", timeout=30)
assert resp.status_code == 200
orders = resp.json()["items"]
assert len(orders) > 0
SAMPLE_ORDER_NUMBER = orders[0]["orderNumber"]
SAMPLE_ORDER_NUMBER

In [ ]:
r = requests.get(f"{BASE_URL}/orders/{SAMPLE_ORDER_NUMBER}", timeout=30)
assert r.status_code == 200
r.json()

## `GET /orderdetails` and lines for one order

In [ ]:
resp = requests.get(
    f"{BASE_URL}/orderdetails",
    params={"orderNumber": SAMPLE_ORDER_NUMBER},
    timeout=30,
)
assert resp.status_code == 200
lines = resp.json()["items"]
assert len(lines) > 0
SAMPLE_PRODUCT_CODE = lines[0]["productCode"]
lines[0]

In [ ]:
r = requests.get(
    f"{BASE_URL}/orders/{SAMPLE_ORDER_NUMBER}/orderdetails",
    timeout=30,
)
assert r.status_code == 200
len(r.json()["items"])

In [ ]:
r = requests.get(
    f"{BASE_URL}/orders/{SAMPLE_ORDER_NUMBER}/orderdetails/{SAMPLE_PRODUCT_CODE}",
    timeout=30,
)
assert r.status_code == 200
r.json()

## Mutating examples (optional)

Creates a throwaway customer, updates it, then deletes it. Requires valid FK targets if you adapt these for orders or order lines.

In [ ]:
payload = {
    "customerName": "Notebook Test Co",
    "contactLastName": "Test",
    "contactFirstName": "Notebook",
    "phone": "555-0100",
    "addressLine1": "1 Notebook Lane",
    "city": "NYC",
    "country": "USA",
}
resp = requests.post(f"{BASE_URL}/customers", json=payload, timeout=30)
assert resp.status_code == 200, resp.text
new_cust_id = int(resp.json())
new_cust_id

In [ ]:
body = {
    "customerName": "Notebook Test Co Updated",
    "contactLastName": "Test",
    "contactFirstName": "Notebook",
    "phone": "555-0100",
    "addressLine1": "1 Notebook Lane",
    "city": "NYC",
    "country": "USA",
}
r = requests.put(f"{BASE_URL}/customers/{new_cust_id}", json=body, timeout=30)
assert r.status_code == 200 and r.json() == {"updated": 1}
requests.get(f"{BASE_URL}/customers/{new_cust_id}", timeout=30).json()

In [ ]:
r = requests.delete(f"{BASE_URL}/customers/{new_cust_id}", timeout=30)
assert r.status_code == 200 and r.json()["deleted"] == 1
gone = requests.get(f"{BASE_URL}/customers/{new_cust_id}", timeout=30)
assert gone.status_code == 404

## Optional: starter `GET /harry-potter` (JSON file backend)

In [ ]:
resp = requests.get(f"{BASE_URL}/harry-potter", timeout=30)
assert resp.status_code == 200
len(resp.json()["items"])